# Deploy and Run the AQC+Trotter Hamiltonian Dynamics Function

This interactive guide shows how to upload the AQC+Trotter Hamiltonian dynamics function to Qiskit Serverless and run an example workload.

This is a quickstart: it deploys the function and runs one small example. For detailed documentation on how to use the function — the full input reference, the output format, and the available execution backends — see the [template README](README.md).


## Requirements

This guide was run last with the following local package versions:

In [1]:
from qiskit import __version__ as qiskit_version
from qiskit_ibm_catalog import __version__ as catalog_version

print("qiskit version:", qiskit_version)
print("qiskit-ibm-catalog version:", catalog_version)

qiskit version: 2.5.1
qiskit-ibm-catalog version: 0.18.0


## 1. Authentication

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key (token) and CRN (instance), which you can find on the [IBM Quantum Platform](https://quantum.cloud.ibm.com) dashboard. This will allow you to locally instantiate the serverless client to upload or run the selected function:

```python
from qiskit_ibm_catalog import QiskitServerless
serverless = QiskitServerless(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")
```

You can optionally use `save_account()` to save your credentials in your local environment (see the [Set up your IBM Cloud account](/docs/guides/cloud-setup#cloud-save) guide). Note that this writes your credentials to the same file as [`QiskitRuntimeService.save_account()`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#save_account):

```python
QiskitServerless.save_account(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")
```

If the account is saved, there is no need to provide the token to authenticate:

In [4]:
from qiskit_ibm_catalog import QiskitServerless

# Authenticate to the remote cluster
# In this case, loading a saved account
serverless = QiskitServerless()

# REPLACE WITH YOUR OWN CREDENTIALS or SAVED ACCOUNT
# serverless = QiskitServerless(channel="ibm_quantum_platform", token="MY_TOKEN", instance="MY_CRN")

## 2. Declare dependencies

Packages the function needs on top of the managed base serverless image.

> The gateway only installs names on its allowlist ([`requirements-dynamic-dependencies.txt`](https://github.com/Qiskit/qiskit-serverless/blob/main/docker-images/requirements-dynamic-dependencies.txt)), matched by package name and pinned to the allowed version with `==`. Anything else must arrive transitively (as a dependency of an allowlisted package). `[extras]` *are* honoured — `qiskit-addon-aqc-tensor[quimb-jax]` is what drags `quimb` / `jax` in here. `cotengrust` is needed for memory efficiency during tensor network simulation. `qiskit-aer` is listed separately for the `fake` backend (local noisy simulation)

In [5]:
DEPENDENCIES = [
    "qiskit-addon-aqc-tensor[quimb-jax]==0.3.1",
    "qiskit-aer==0.17.2",
    "cotengrust==0.2.0",
]

## 3. Define and upload the function


In [6]:
from qiskit_ibm_catalog import QiskitFunction

fn = QiskitFunction(
    title="aqc-dynamics-function",
    entrypoint="program.py",
    working_dir="source_files/",
    dependencies=DEPENDENCIES,
)
serverless.upload(fn)

QiskitFunction(aqc-dynamics-function)

## 4. Verify it registered

In [7]:
next(p for p in serverless.list() if p.title == "aqc-dynamics-function")

QiskitFunction(aqc-dynamics-function)

## 5. Run it (Small Test Run)


In [8]:
from qiskit.quantum_info import SparsePauliOp

fn = serverless.load("aqc-dynamics-function")

n = 8
H = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, i + 1], 1.0) for i in range(n - 1)] + [("X", [i], 0.8) for i in range(n)],
    num_qubits=n,
)

# Varied-ansatz AQC compression: the first 4 steps compress into a shallow
# 1-layer ansatz, the next 2 into a deeper 2-layer ansatz (which holds fidelity
# for the later, more-entangled states); the remaining steps run as plain Trotter.
job = fn.run(
    t_steps=8,
    aqc_segments=[
        {"n_steps": 4, "ansatz_steps": 1},
        {"n_steps": 2, "ansatz_steps": 2},
    ],
    hamiltonian=H,
    aqc_options=dict(max_bond=32),
    backend="statevector",
)
print(job.job_id)

77f87802-b5fc-4bc9-a00a-5e4ecfb7b858


## 6. Logs and result (From Test Run)

`status()` reports both the coarse job lifecycle **and** the per-stage sub-status the
function publishes as it runs (via `update_status`):

`QUEUED → INITIALIZING → RUNNING: OPTIMIZING_FOR_HARDWARE → RUNNING: WAITING_FOR_QPU → RUNNING: EXECUTING_QPU → RUNNING: POST_PROCESSING → DONE`

| `status()` value | stage |
|---|---|
| `RUNNING: OPTIMIZING_FOR_HARDWARE` | state-prep + Trotter build + AQC compression |
| `RUNNING: WAITING_FOR_QPU` | job sits in the QPU queue — `runtime` backend only |
| `RUNNING: EXECUTING_QPU` | circuits executing (local sims mark this directly) |
| `RUNNING: POST_PROCESSING` | assembling the result dict |

Terminal states are `DONE`, `ERROR`, or `CANCELED`. This `statevector` run has no QPU queue,
so it skips `RUNNING: WAITING_FOR_QPU`. Fetch the result once it is `DONE`.

In [16]:
job.status()

'DONE'

In [17]:
print(job.logs())

2026-08-06 20:24:03,802	INFO job_manager.py:583 -- Runtime env is setting up.
Running entrypoint for job raysubmit_XJkBm2HVxNpFvcKu: python program.py
INFO: Pass: ContainsInstruction - 0.01168 (ms)
INFO: Pass: InverseCancellation - 0.02432 (ms)
INFO: Pass: ContractIdleWiresInControlFlow - 0.00286 (ms)
INFO: Pass: UnitarySynthesis - 0.00405 (ms)
INFO: Pass: HighLevelSynthesis - 0.00525 (ms)
INFO: Pass: BasisTranslator - 0.02456 (ms)
INFO: Pass: Size - 0.00477 (ms)
INFO: Pass: Depth - 0.01907 (ms)
INFO: Pass: FixedPoint - 0.01025 (ms)
INFO: Pass: FixedPoint - 0.00381 (ms)
INFO: Pass: Optimize1qGatesDecomposition - 2.21872 (ms)
INFO: Pass: InverseCancellation - 0.00739 (ms)
INFO: Pass: ContractIdleWiresInControlFlow - 0.00238 (ms)
INFO: Pass: GatesInBasis - 0.00477 (ms)
INFO: Pass: Size - 0.00405 (ms)
INFO: Pass: Depth - 0.00453 (ms)
INFO: Pass: FixedPoint - 0.00691 (ms)
INFO: Pass: FixedPoint - 0.00310 (ms)
INFO: Pass: ContainsInstruction - 0.00358 (ms)
INFO: Total Transpile Time - 35.47

In [18]:
result = job.result()
print("observables:", result["observable_labels"])
print("times:", result["times"])
print("AQC fidelities:", result["metadata"]["aqc_fidelities"])

observables: ['Z_0', 'Z_1', 'Z_2', 'Z_3', 'Z_4', 'Z_5', 'Z_6', 'Z_7']
times: [0.0, 0.2, 0.4, 0.6000000000000001, 0.8, 1.0, 1.2000000000000002, 1.4000000000000001, 1.6]
AQC fidelities: {'1': 0.9999964237244967, '2': 0.9999656680192857, '3': 0.9999835491853446, '4': 0.9999698402771173, '5': 0.9999799729393999, '6': 0.9999387273637694}
